# Pipeline inspection — 3-stock dataset

Questo notebook ispeziona tutti i dataset intermedi prodotti dalla pipeline a 3 stock (NVDA, PLTR, AMD).

**Datasets ispezionati (in ordine di pipeline):**

| Stage | File | Descrizione |
|---|---|---|
| 1b | `data/extraction/wsb_comments_3stocks.csv` | Filtro 3-stock dal grezzo |
| 2  | `data/extraction/wsb_comments_2025_clean_structural.csv` | Cleaning strutturale |
| 3  | `data/extraction/wsb_submissions_2025_recovered.csv` | Submissions recuperate |
| 4  | `data/extraction/wsb_merged_comments_with_submission.csv` | Dataset principale (merged) |
| 5  | `data/summarization/thread_chunks_v1.csv` | Chunks per il summarizer |

**Check effettuati per ogni dataset:**
- Dimensioni (righe, colonne)
- Schema colonne e tipi
- Distribuzione ticker (verifica che siano solo 3 stock)
- Valori nulli
- Duplicati
- Consistenza con lo stage precedente

**Check trasversali finali:**
- Join key integrity: tutti i `submission_id` dei commenti esistono nelle submissions
- Copertura chunks: tutti i thread del merged sono presenti nei chunks
- Formula join chunk → commento: `chunk_id = (thread_position - 1) // 100`

In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', '{:,.2f}'.format)

ROOT = Path('..')  # run from notebooks/

PATHS = {
    'interim_3stocks':  ROOT / 'data/extraction/wsb_comments_3stocks.csv',
    'clean':            ROOT / 'data/extraction/wsb_comments_2025_clean_structural.csv',
    'submissions':      ROOT / 'data/extraction/wsb_submissions_2025_recovered.csv',
    'merged':           ROOT / 'data/extraction/wsb_merged_comments_with_submission.csv',
    'chunks':           ROOT / 'data/summarization/thread_chunks_v1.csv',
}

TARGET_TICKERS = {'NVDA', 'PLTR', 'AMD'}

def file_size_mb(path):
    return os.path.getsize(path) / 1024**2

def contains_target_ticker(ticker_str):
    return bool(set(str(ticker_str).split('|')) & TARGET_TICKERS)

print('Setup OK. Paths:')
for k, p in PATHS.items():
    exists = p.exists()
    size = f'{file_size_mb(p):.1f} MB' if exists else 'MISSING'
    print(f'  {k:20s} → {size}')

Setup OK. Paths:
  interim_3stocks      → 106.1 MB
  clean                → 101.4 MB
  submissions          → 5.6 MB
  merged               → 98.9 MB
  chunks               → 31.8 MB


---
## Stage 1b — `wsb_comments_3stocks.csv`
Output di `filter_to_3stocks.py`. Punto di partenza della pipeline 3-stock.

In [2]:
df_interim = pd.read_csv(PATHS['interim_3stocks'], low_memory=False)

print(f'Shape: {df_interim.shape[0]:,} righe × {df_interim.shape[1]} colonne')
print(f'File size: {file_size_mb(PATHS["interim_3stocks"]):.1f} MB')
print()
print('Colonne e tipi:')
print(df_interim.dtypes.to_string())

Shape: 248,072 righe × 21 colonne
File size: 106.1 MB

Colonne e tipi:
id                       object
created_utc             float64
date_utc                 object
source_type              object
subreddit                object
author                   object
score                     int64
title                   float64
body_text                object
raw_text                 object
permalink                object
link_id                  object
parent_id                object
submission_id            object
matched_tickers          object
matched_terms            object
match_sources            object
match_count               int64
is_multi_match            int64
match_confidence         object
needs_context_filter      int64


In [3]:
print('=== Valori nulli ===')
nulls = df_interim.isnull().sum()
print(nulls[nulls > 0].to_string() if nulls.any() else 'Nessun valore nullo.')
print()

print('=== Duplicati (comment id) ===')
dup_ids = df_interim['id'].duplicated().sum()
print(f'Righe con id duplicato: {dup_ids:,}')

=== Valori nulli ===
title    248072

=== Duplicati (comment id) ===
Righe con id duplicato: 0


In [4]:
print('=== Distribuzione matched_tickers (top 20) ===')
print(df_interim['matched_tickers'].value_counts().head(20).to_string())
print()

# Verifica: nessun ticker fuori dal target deve essere SOLO fuori-target
# (commenti con AMD|TSLA sono accettabili perché contengono AMD)
non_target_only = df_interim[~df_interim['matched_tickers'].apply(contains_target_ticker)]
print(f'⚠️  Righe senza nessun ticker target: {len(non_target_only):,}  ← deve essere 0')

=== Distribuzione matched_tickers (top 20) ===
matched_tickers
NVDA              149642
PLTR               37161
AMD                36438
AMD|NVDA            5289
NVDA|TSLA           4174
PLTR|TSLA           1966
MSFT|NVDA           1655
NVDA|PLTR           1628
AMD|INTC            1377
INTC|NVDA           1360
GOOGL|NVDA          1022
AMD|INTC|NVDA        763
AMD|PLTR             492
AAPL|NVDA            427
AMD|TSLA             351
NVDA|PLTR|TSLA       274
META|NVDA            238
MSFT|NVDA|TSLA       228
AMD|MSFT             157
AMD|NVDA|TSLA        154

⚠️  Righe senza nessun ticker target: 0  ← deve essere 0


In [5]:
print('=== Range temporale ===')
df_interim['date_utc'] = pd.to_datetime(df_interim['date_utc'])
print(f'Da: {df_interim["date_utc"].min()}')
print(f'A:  {df_interim["date_utc"].max()}')
print()

print('=== match_confidence distribution ===')
print(df_interim['match_confidence'].value_counts().to_string())
print()

print('=== source_type distribution ===')
print(df_interim['source_type'].value_counts().to_string())

=== Range temporale ===
Da: 2025-01-01 00:00:00
A:  2025-12-31 00:00:00

=== match_confidence distribution ===
match_confidence
high           223241
multi_stock     24831

=== source_type distribution ===
source_type
comment    248072


---
## Stage 2 — `wsb_comments_2025_clean_structural.csv`
Output di `clean_comments_structural.py`. Deve contenere solo i 3 stock e avere la colonna `thread_position`.

In [6]:
df_clean = pd.read_csv(PATHS['clean'], low_memory=False)

print(f'Shape: {df_clean.shape[0]:,} righe × {df_clean.shape[1]} colonne')
print(f'File size: {file_size_mb(PATHS["clean"]):.1f} MB')
print()

# Confronto con stage 1b
removed = df_interim.shape[0] - df_clean.shape[0]
print(f'Righe rimosse dal cleaning: {removed:,} ({removed/df_interim.shape[0]*100:.1f}%)')
print()

print('Colonne:')
print(list(df_clean.columns))
print()
assert 'thread_position' in df_clean.columns, '❌ MANCA thread_position!'
print('✅ thread_position presente')

Shape: 239,080 righe × 22 colonne
File size: 101.4 MB

Righe rimosse dal cleaning: 8,992 (3.6%)

Colonne:
['id', 'created_utc', 'date_utc', 'source_type', 'subreddit', 'author', 'score', 'title', 'body_text', 'raw_text', 'permalink', 'link_id', 'parent_id', 'submission_id', 'matched_tickers', 'matched_terms', 'match_sources', 'match_count', 'is_multi_match', 'match_confidence', 'needs_context_filter', 'thread_position']

✅ thread_position presente


In [7]:
print('=== Verifica 3-stock filter (distribuzione matched_tickers, top 20) ===')
print(df_clean['matched_tickers'].value_counts().head(20).to_string())
print()

non_target_clean = df_clean[~df_clean['matched_tickers'].apply(contains_target_ticker)]
print(f'⚠️  Righe senza ticker target: {len(non_target_clean):,}  ← deve essere 0')

all_tickers_in_clean = set()
for val in df_clean['matched_tickers'].unique():
    for t in str(val).split('|'):
        all_tickers_in_clean.add(t)
extra_tickers = all_tickers_in_clean - TARGET_TICKERS
print(f'Ticker presenti oltre ai 3 target: {extra_tickers}  ← OK se sono solo in combo multi-ticker')

=== Verifica 3-stock filter (distribuzione matched_tickers, top 20) ===
matched_tickers
NVDA              143346
PLTR               35907
AMD                35229
AMD|NVDA            5256
NVDA|TSLA           4122
PLTR|TSLA           1956
MSFT|NVDA           1644
NVDA|PLTR           1613
AMD|INTC            1354
INTC|NVDA           1338
GOOGL|NVDA          1013
AMD|INTC|NVDA        755
AMD|PLTR             487
AAPL|NVDA            421
AMD|TSLA             351
NVDA|PLTR|TSLA       272
META|NVDA            235
MSFT|NVDA|TSLA       228
AMD|MSFT             156
AMD|NVDA|TSLA        154

⚠️  Righe senza ticker target: 0  ← deve essere 0
Ticker presenti oltre ai 3 target: {'TSLA', 'AAPL', 'AMZN', 'GOOGL', 'MSFT', 'META', 'INTC'}  ← OK se sono solo in combo multi-ticker


In [8]:
print('=== Valori nulli ===')
nulls = df_clean.isnull().sum()
print(nulls[nulls > 0].to_string() if nulls.any() else 'Nessun valore nullo.')
print()

print('=== Duplicati (comment id) ===')
dup_ids = df_clean['id'].duplicated().sum()
print(f'Righe con id duplicato: {dup_ids:,}  ← deve essere 0')
print()

print('=== thread_position: check integrità ===')
tp_min = df_clean['thread_position'].min()
tp_null = df_clean['thread_position'].isnull().sum()
print(f'Minimo thread_position: {tp_min}  ← deve essere 1')
print(f'thread_position nulli: {tp_null}  ← deve essere 0')

=== Valori nulli ===
title    239080

=== Duplicati (comment id) ===
Righe con id duplicato: 0  ← deve essere 0

=== thread_position: check integrità ===
Minimo thread_position: 1  ← deve essere 1
thread_position nulli: 0  ← deve essere 0


In [9]:
print('=== submission_id: check integrità ===')
missing_sub = df_clean['submission_id'].isnull().sum()
print(f'submission_id nulli: {missing_sub}  ← deve essere 0')
print(f'Unique submission_id: {df_clean["submission_id"].nunique():,}')
print()

print('=== Commenti per thread (distribuzione) ===')
comments_per_thread = df_clean.groupby('submission_id').size()
print(comments_per_thread.describe().to_string())
print(f'Thread con >100 commenti (multi-chunk): {(comments_per_thread > 100).sum():,}')

=== submission_id: check integrità ===
submission_id nulli: 0  ← deve essere 0
Unique submission_id: 8,740

=== Commenti per thread (distribuzione) ===
count   8,740.00
mean       27.35
std       122.63
min         1.00
25%         1.00
50%         2.00
75%         5.00
max     2,822.00
Thread con >100 commenti (multi-chunk): 512


---
## Stage 3 — `wsb_submissions_2025_recovered.csv`
Submissions recuperate da `recover_parent_submissions.py`.

In [10]:
df_subs = pd.read_csv(PATHS['submissions'], low_memory=False)

print(f'Shape: {df_subs.shape[0]:,} righe × {df_subs.shape[1]} colonne')
print(f'File size: {file_size_mb(PATHS["submissions"]):.1f} MB')
print()
print('Colonne:')
print(list(df_subs.columns))

Shape: 8,740 righe × 21 colonne
File size: 5.6 MB

Colonne:
['id', 'created_utc', 'date_utc', 'source_type', 'subreddit', 'author', 'score', 'title', 'body_text', 'raw_text', 'permalink', 'link_id', 'parent_id', 'submission_id', 'matched_tickers', 'matched_terms', 'match_sources', 'match_count', 'is_multi_match', 'match_confidence', 'needs_context_filter']


In [11]:
print('=== Copertura: submission_id dei commenti vs submissions recuperate ===')
required_sub_ids = set(df_clean['submission_id'].dropna().unique())
recovered_sub_ids = set(df_subs['id'].dropna().unique())

covered = required_sub_ids & recovered_sub_ids
missing = required_sub_ids - recovered_sub_ids

print(f'Submission_id richiesti dai commenti: {len(required_sub_ids):,}')
print(f'Submission_id recuperati: {len(recovered_sub_ids):,}')
print(f'Copertura: {len(covered):,} ({len(covered)/len(required_sub_ids)*100:.1f}%)')
print(f'⚠️  submission_id mancanti nelle submissions: {len(missing):,}')
if missing:
    print(f'   Esempi: {list(missing)[:5]}')

=== Copertura: submission_id dei commenti vs submissions recuperate ===
Submission_id richiesti dai commenti: 8,740
Submission_id recuperati: 8,740
Copertura: 8,740 (100.0%)
⚠️  submission_id mancanti nelle submissions: 0


In [12]:
print('=== Valori nulli nelle colonne chiave ===')
key_cols_subs = ['id', 'title', 'body_text', 'created_utc']
for col in key_cols_subs:
    if col in df_subs.columns:
        n = df_subs[col].isnull().sum()
        print(f'  {col}: {n:,} nulli')
print()

print('=== Duplicati (submission id) ===')
dup_subs = df_subs['id'].duplicated().sum()
print(f'Submission id duplicati: {dup_subs:,}  ← deve essere 0')

=== Valori nulli nelle colonne chiave ===
  id: 0 nulli
  title: 0 nulli
  body_text: 2,544 nulli
  created_utc: 0 nulli

=== Duplicati (submission id) ===
Submission id duplicati: 0  ← deve essere 0


---
## Stage 4 — `wsb_merged_comments_with_submission.csv` (dataset principale)
Output di `build_merged_dataset.py`. Una riga per commento, arricchita con `submission_text`.

In [13]:
df_merged = pd.read_csv(PATHS['merged'], low_memory=False)

print(f'Shape: {df_merged.shape[0]:,} righe × {df_merged.shape[1]} colonne')
print(f'File size: {file_size_mb(PATHS["merged"]):.1f} MB')
print()

# Confronto con stage 2
diff = df_clean.shape[0] - df_merged.shape[0]
print(f'Righe in clean: {df_clean.shape[0]:,}')
print(f'Righe in merged: {df_merged.shape[0]:,}')
print(f'Differenza: {diff:,}  ← deve essere 0 (left join, nessuna riga persa)')
print()
print('Colonne:')
print(list(df_merged.columns))

Shape: 239,080 righe × 11 colonne
File size: 98.9 MB

Righe in clean: 239,080
Righe in merged: 239,080
Differenza: 0  ← deve essere 0 (left join, nessuna riga persa)

Colonne:
['id', 'submission_id', 'thread_position', 'created_utc', 'author', 'score', 'message_text', 'submission_text', 'matched_tickers', 'match_count', 'needs_context_filter']


In [14]:
print('=== Verifica 3-stock filter nel merged ===')
print(df_merged['matched_tickers'].value_counts().head(20).to_string())
print()

non_target_merged = df_merged[~df_merged['matched_tickers'].apply(contains_target_ticker)]
print(f'⚠️  Righe senza ticker target: {len(non_target_merged):,}  ← deve essere 0')

=== Verifica 3-stock filter nel merged ===
matched_tickers
NVDA              143346
PLTR               35907
AMD                35229
AMD|NVDA            5256
NVDA|TSLA           4122
PLTR|TSLA           1956
MSFT|NVDA           1644
NVDA|PLTR           1613
AMD|INTC            1354
INTC|NVDA           1338
GOOGL|NVDA          1013
AMD|INTC|NVDA        755
AMD|PLTR             487
AAPL|NVDA            421
AMD|TSLA             351
NVDA|PLTR|TSLA       272
META|NVDA            235
MSFT|NVDA|TSLA       228
AMD|MSFT             156
AMD|NVDA|TSLA        154

⚠️  Righe senza ticker target: 0  ← deve essere 0


In [15]:
print('=== Valori nulli ===')
nulls = df_merged.isnull().sum()
print(nulls[nulls > 0].to_string() if nulls.any() else 'Nessun valore nullo.')
print()

print('=== submission_text vuoto o nullo ===')
empty_sub_text = df_merged['submission_text'].isnull().sum()
blank_sub_text = (df_merged['submission_text'].fillna('').str.strip() == '').sum()
print(f'submission_text nullo: {empty_sub_text:,}')
print(f'submission_text vuoto/blank: {blank_sub_text:,}')
print()

print('=== message_text vuoto o nullo ===')
empty_msg = df_merged['message_text'].isnull().sum()
blank_msg = (df_merged['message_text'].fillna('').str.strip() == '').sum()
print(f'message_text nullo: {empty_msg:,}')
print(f'message_text vuoto/blank: {blank_msg:,}')

=== Valori nulli ===
Nessun valore nullo.

=== submission_text vuoto o nullo ===
submission_text nullo: 0
submission_text vuoto/blank: 0

=== message_text vuoto o nullo ===
message_text nullo: 0
message_text vuoto/blank: 0


In [16]:
print('=== Duplicati (comment id) ===')
dup_merged = df_merged['id'].duplicated().sum()
print(f'Comment id duplicati: {dup_merged:,}  ← deve essere 0')
print()

print('=== thread_position integrità ===')
print(f'Minimo: {df_merged["thread_position"].min()}  ← deve essere 1')
print(f'Nulli: {df_merged["thread_position"].isnull().sum()}  ← deve essere 0')
print()

print('=== Unique threads nel merged ===')
print(f'Unique submission_id: {df_merged["submission_id"].nunique():,}')
print()

print('=== Sample di 3 righe (colonne chiave) ===')
df_merged[['id','submission_id','thread_position','matched_tickers','message_text','submission_text']].sample(3, random_state=42).assign(
    message_text=lambda x: x['message_text'].str[:80],
    submission_text=lambda x: x['submission_text'].str[:80]
)

=== Duplicati (comment id) ===
Comment id duplicati: 0  ← deve essere 0

=== thread_position integrità ===
Minimo: 1  ← deve essere 1
Nulli: 0  ← deve essere 0

=== Unique threads nel merged ===
Unique submission_id: 8,740

=== Sample di 3 righe (colonne chiave) ===


,id,submission_id,thread_position,matched_tickers,message_text,submission_text
224535,nqne1hv,1p5vmzy,2,GOOGL|PLTR,"Trust me dude, everyone is always pussy and wants to sell early. It’s what t...",Sundar is my daddy\n\nFull ported back in Aug/Sep and it was the best thing ...
84958,mjle7h9,1jj0773,77,NVDA,Sold 5k in nvidia puts @ 3:59,"What Are Your Moves Tomorrow, March 25, 2025\n\nThis post contains content n..."
88379,mkvgnqk,1jorule,99,AMD,Back to normal market- everything is green except AMD,"Daily Discussion Thread for April 01, 2025\n\nThis post contains content not..."


---
## Stage 5 — `thread_chunks_v1.csv`
Output di `prepare_threads.py`. Una riga per chunk, pronto per il summarizer.

In [17]:
df_chunks = pd.read_csv(PATHS['chunks'], low_memory=False)

print(f'Shape: {df_chunks.shape[0]:,} righe × {df_chunks.shape[1]} colonne')
print(f'File size: {file_size_mb(PATHS["chunks"]):.1f} MB')
print()
print('Colonne:')
print(list(df_chunks.columns))

Shape: 10,481 righe × 5 colonne
File size: 31.8 MB

Colonne:
['submission_id', 'chunk_id', 'n_messages_in_chunk', 'submission_text', 'chunk_comments']


In [18]:
print('=== Unique threads nei chunks ===')
threads_in_chunks = df_chunks['submission_id'].nunique()
threads_in_merged = df_merged['submission_id'].nunique()
print(f'Threads nel merged: {threads_in_merged:,}')
print(f'Threads nei chunks: {threads_in_chunks:,}')
print(f'⚠️  Differenza: {threads_in_merged - threads_in_chunks:,}  ← deve essere 0')
print()

missing_threads = set(df_merged['submission_id'].unique()) - set(df_chunks['submission_id'].unique())
extra_threads   = set(df_chunks['submission_id'].unique()) - set(df_merged['submission_id'].unique())
print(f'Thread nel merged ma non nei chunks: {len(missing_threads)}')
print(f'Thread nei chunks ma non nel merged: {len(extra_threads)}')

=== Unique threads nei chunks ===
Threads nel merged: 8,740
Threads nei chunks: 8,740
⚠️  Differenza: 0  ← deve essere 0

Thread nel merged ma non nei chunks: 0
Thread nei chunks ma non nel merged: 0


In [19]:
print('=== Distribuzione chunk_id (quanti chunk per thread) ===')
chunks_per_thread = df_chunks.groupby('submission_id')['chunk_id'].max() + 1
print(chunks_per_thread.describe().to_string())
print()
print('Distribuzione numero di chunk:')
print(chunks_per_thread.value_counts().sort_index().head(15).to_string())
print(f'Thread con >1 chunk (multi-chunk): {(chunks_per_thread > 1).sum():,}')

=== Distribuzione chunk_id (quanti chunk per thread) ===
count   8,740.00
mean        1.20
std         1.16
min         1.00
25%         1.00
50%         1.00
75%         1.00
max        29.00

Distribuzione numero di chunk:
chunk_id
1     8228
2      176
3      103
4       74
5       47
6       29
7       21
8       16
9       12
10       6
11       1
12       4
14       7
15       1
16       7
Thread con >1 chunk (multi-chunk): 512


In [20]:
print('=== n_messages_in_chunk ===')
print(df_chunks['n_messages_in_chunk'].describe().to_string())
print()
print(f'Chunk con < 100 messaggi (ultimo chunk di thread lunghi): '
      f'{(df_chunks["n_messages_in_chunk"] < 100).sum():,}')
print(f'Chunk con = 100 messaggi: '
      f'{(df_chunks["n_messages_in_chunk"] == 100).sum():,}')
print()

total_messages_in_chunks = df_chunks['n_messages_in_chunk'].sum()
print(f'Totale messaggi contati nei chunks: {total_messages_in_chunks:,}')
print(f'Totale commenti nel merged: {df_merged.shape[0]:,}')
print(f'⚠️  Differenza: {df_merged.shape[0] - total_messages_in_chunks:,}  ← deve essere 0')

=== n_messages_in_chunk ===
count   10,481.00
mean        22.81
std         37.32
min          1.00
25%          1.00
50%          2.00
75%         18.00
max        100.00

Chunk con < 100 messaggi (ultimo chunk di thread lunghi): 8,738
Chunk con = 100 messaggi: 1,743

Totale messaggi contati nei chunks: 239,080
Totale commenti nel merged: 239,080
⚠️  Differenza: 0  ← deve essere 0


In [21]:
print('=== Valori nulli nei chunks ===')
nulls = df_chunks.isnull().sum()
print(nulls[nulls > 0].to_string() if nulls.any() else 'Nessun valore nullo.')
print()

print('=== chunk_id duplicati per lo stesso submission_id ===')
dup_chunks = df_chunks.duplicated(subset=['submission_id', 'chunk_id']).sum()
print(f'(submission_id, chunk_id) duplicati: {dup_chunks:,}  ← deve essere 0')

=== Valori nulli nei chunks ===
Nessun valore nullo.

=== chunk_id duplicati per lo stesso submission_id ===
(submission_id, chunk_id) duplicati: 0  ← deve essere 0


---
## Check trasversali — consistenza dell'intera pipeline

In [22]:
print('=' * 60)
print('CHECK 1: Riduzione dimensionale stage-by-stage')
print('=' * 60)

stages = [
    ('1b interim_3stocks', df_interim.shape[0]),
    ('2  clean',           df_clean.shape[0]),
    ('4  merged',          df_merged.shape[0]),
]
for name, n in stages:
    pct_of_interim = n / df_interim.shape[0] * 100
    print(f'  Stage {name}: {n:>10,} righe  ({pct_of_interim:.1f}% del grezzo)')

CHECK 1: Riduzione dimensionale stage-by-stage
  Stage 1b interim_3stocks:    248,072 righe  (100.0% del grezzo)
  Stage 2  clean:    239,080 righe  (96.4% del grezzo)
  Stage 4  merged:    239,080 righe  (96.4% del grezzo)


In [23]:
print('=' * 60)
print('CHECK 2: Join key integrity — submission_id')
print('='*60)

# Tutti i submission_id dei commenti devono esistere nelle submissions
merged_sub_ids  = set(df_merged['submission_id'].dropna().unique())
subs_ids        = set(df_subs['id'].dropna().unique())
orphan_comments = merged_sub_ids - subs_ids

print(f'submission_id unici nei commenti:  {len(merged_sub_ids):,}')
print(f'id unici nelle submissions:        {len(subs_ids):,}')
print(f'submission_id orfani (no parent):  {len(orphan_comments):,}  ← idealmente 0')
if orphan_comments:
    print(f'  Esempi: {list(orphan_comments)[:5]}')

CHECK 2: Join key integrity — submission_id
submission_id unici nei commenti:  8,740
id unici nelle submissions:        8,740
submission_id orfani (no parent):  0  ← idealmente 0


In [24]:
print('=' * 60)
print('CHECK 3: Formula join chunk → commento')
print('         chunk_id = (thread_position - 1) // 100')
print('=' * 60)

# Calcola chunk_id atteso per ogni commento
df_merged['expected_chunk_id'] = (df_merged['thread_position'] - 1) // 100

# Verifica che ogni (submission_id, chunk_id) esista nei chunks
chunks_keys = set(zip(df_chunks['submission_id'], df_chunks['chunk_id']))
merged_keys = set(zip(df_merged['submission_id'], df_merged['expected_chunk_id']))

unmatched = merged_keys - chunks_keys
print(f'Coppie (submission_id, chunk_id) nel merged: {len(merged_keys):,}')
print(f'Coppie (submission_id, chunk_id) nei chunks: {len(chunks_keys):,}')
print(f'Coppie nel merged senza corrispondenza nei chunks: {len(unmatched):,}  ← deve essere 0')

# Cleanup colonna temporanea
df_merged.drop(columns=['expected_chunk_id'], inplace=True)

CHECK 3: Formula join chunk → commento
         chunk_id = (thread_position - 1) // 100
Coppie (submission_id, chunk_id) nel merged: 10,481
Coppie (submission_id, chunk_id) nei chunks: 10,481
Coppie nel merged senza corrispondenza nei chunks: 0  ← deve essere 0


In [25]:
print('=' * 60)
print('RIEPILOGO FINALE')
print('=' * 60)

summary = {
    'Stage 1b — interim_3stocks': {
        'righe': df_interim.shape[0],
        'colonne': df_interim.shape[1],
        'ticker fuori target': (~df_interim['matched_tickers'].apply(contains_target_ticker)).sum(),
        'id duplicati': df_interim['id'].duplicated().sum(),
    },
    'Stage 2 — clean_structural': {
        'righe': df_clean.shape[0],
        'colonne': df_clean.shape[1],
        'ticker fuori target': (~df_clean['matched_tickers'].apply(contains_target_ticker)).sum(),
        'id duplicati': df_clean['id'].duplicated().sum(),
        'thread_position min': df_clean['thread_position'].min(),
    },
    'Stage 3 — submissions': {
        'righe': df_subs.shape[0],
        'copertura submission_id': f"{len(set(df_clean['submission_id'].unique()) & set(df_subs['id'].unique())):,} / {df_clean['submission_id'].nunique():,}",
    },
    'Stage 4 — merged': {
        'righe': df_merged.shape[0],
        'colonne': df_merged.shape[1],
        'ticker fuori target': (~df_merged['matched_tickers'].apply(contains_target_ticker)).sum(),
        'id duplicati': df_merged['id'].duplicated().sum(),
        'submission_text nulli': df_merged['submission_text'].isnull().sum(),
        'message_text nulli': df_merged['message_text'].isnull().sum(),
    },
    'Stage 5 — chunks': {
        'righe (chunks)': df_chunks.shape[0],
        'thread unici': df_chunks['submission_id'].nunique(),
        'totale messaggi': df_chunks['n_messages_in_chunk'].sum(),
        '(sub_id, chunk_id) duplicati': df_chunks.duplicated(subset=['submission_id','chunk_id']).sum(),
    },
}

for stage, metrics in summary.items():
    print(f'\n{stage}:')
    for k, v in metrics.items():
        flag = '✅' if v == 0 or v == 1 else ''
        print(f'  {k}: {v:,}' if isinstance(v, int) else f'  {k}: {v}')

RIEPILOGO FINALE

Stage 1b — interim_3stocks:
  righe: 248,072
  colonne: 21
  ticker fuori target: 0
  id duplicati: 0

Stage 2 — clean_structural:
  righe: 239,080
  colonne: 22
  ticker fuori target: 0
  id duplicati: 0
  thread_position min: 1

Stage 3 — submissions:
  righe: 8,740
  copertura submission_id: 8,740 / 8,740

Stage 4 — merged:
  righe: 239,080
  colonne: 11
  ticker fuori target: 0
  id duplicati: 0
  submission_text nulli: 0
  message_text nulli: 0

Stage 5 — chunks:
  righe (chunks): 10,481
  thread unici: 8,740
  totale messaggi: 239080
  (sub_id, chunk_id) duplicati: 0
